# 03 · Isomorfismos y tipos algebraicos de datos

> **En el examen:** es el **Ejercicio 2** (entre 1 y 2 puntos). Sale en todos los exámenes
> desde 20-21. Te dan dos tipos y te piden demostrar que son **isomorfos**, normalmente de
> dos formas:
> - **a)** con **aritmética** (cardinalidades): una cadena de igualdades (≈ 0,5 puntos),
> - **b)** con **código**: implementar `from` y `to` en una clase que extiende `Isomorphic`
>   (≈ 1,5 puntos).

Necesitas del archivo 01: tuplas, `Option`, `Either`, `Unit`, `Boolean`, lambdas,
funciones hechas de `case` y `trait`.

---

### Preámbulo
Ejecuta la siguiente celda antes que las demás. Las celdas de código se pueden ejecutar en orden; los fragmentos que no son ejecutables por sí solos se muestran como texto.

In [ ]:
// Preámbulo: ejecuta esta celda primero (las mismas definiciones que vienen en los exámenes)
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)
    def law2(b: B): Boolean = equalB(from(to(b)), b)

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)
    def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
        tree match
            case Empty() => empty
            case Node(left, a, right) => node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

import Tree._

def pow(base: Int, exp: Int): Int = math.pow(base, exp).toInt

## 1. ¿Qué es un isomorfismo?

Dos tipos `A` y `B` son **isomorfos** si puedes convertir cualquier valor de `A` en uno de
`B` y volver, **sin perder información**. Es decir, existen dos funciones:

- `from: A => B`
- `to: B => A`

que son **inversas**: si vas y vuelves, te quedas como estabas.

- `to(from(a)) == a` para todo `a` (**ley 1**)
- `from(to(b)) == b` para todo `b` (**ley 2**)

Ejemplo intuitivo: `(Int, String)` y `(String, Int)` son isomorfos: basta con dar la vuelta
al par. No se pierde nada.

En el preámbulo del examen viene esta plantilla:

In [ ]:
trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A

    def law1(a: A): Boolean = to(from(a)) == a
    def law2(b: B): Boolean = from(to(b)) == b

Tú solo implementas `from` y `to`. Las leyes ya están escritas y las usan los tests.

---

## 2. ¿Por qué se llaman "algebraicos"? Contar valores

La **cardinalidad** `|A|` es **cuántos valores distintos** tiene el tipo `A`.

| Tipo | Cardinalidad | Por qué |
|---|---|---|
| `Nothing` | 0 | no tiene valores |
| `Unit` | 1 | solo `()` |
| `Boolean` | 2 | `true`, `false` |
| `Either[A, B]` | \|A\| **+** \|B\| | o un `Left(a)` o un `Right(b)` (**suma**) |
| `(A, B)` | \|A\| **·** \|B\| | todas las combinaciones (**producto**) |
| `(A, B, C)` | \|A\| · \|B\| · \|C\| | |
| `A => B` | \|B\| **^** \|A\| | por cada valor de `A` eliges uno de `B` (**potencia**) |
| `Option[A]` | \|A\| + 1 | `Some(a)` o `None` |
| `List[A]` | 1 + \|A\| · \|List[A]\| | o `Nil` o cabeza y resto |

Ojo con la potencia: **la base es el resultado, el exponente es el argumento**.
`Boolean => Int` es `|Int|^2` (para `false` eliges un Int y para `true` otro).

**Regla de oro:** dos tipos son isomorfos **si y solo si** sus fórmulas son iguales usando
las reglas del álgebra de toda la vida.

### Las igualdades del álgebra que vas a usar

| Álgebra | En tipos (lo que significa) |
|---|---|
| a + b = b + a | `Either[A, B]` ≅ `Either[B, A]` |
| a · b = b · a | `(A, B)` ≅ `(B, A)` |
| a · (b + c) = a·b + a·c | `(A, Either[B, C])` ≅ `Either[(A, B), (A, C)]` |
| a · 1 = a | `(A, Unit)` ≅ `A` |
| a + 0 = a | `Either[A, Nothing]` ≅ `A` |
| a^1 = a | `Unit => A` ≅ `A` |
| a^0 = 1 | `Nothing => A` ≅ `Unit` |
| a^(b+c) = a^b · a^c | `Either[B, C] => A` ≅ `(B => A, C => A)` |
| (a·b)^c = a^c · b^c | `C => (A, B)` ≅ `(C => A, C => B)` |
| a^(b·c) = (a^c)^b | `((B, C)) => A` ≅ `B => C => A` (¡es la currificación!) |
| (a+b)² = a² + 2ab + b² | `Boolean => Either[A, B]` ≅ … (ver ejemplo 4) |

---

## 3. Apartado a): la cadena de igualdades

Escribe una igualdad por línea, desde el primer tipo hasta el segundo. En cada paso aplica
**una** regla. Formato que usa el profesor:

```
|Option[A] => B| =
|B| ^ |Option[A]| =
|B| ^ (1 + |A|) =
|B| ^ 1 * |B| ^ |A| =
|B| * |B| ^ |A| =
|B| * |A => B| =
|(B, A => B)|
```

Consejo: desarrolla **los dos lados** hasta una forma "plana" (sumas de productos) y
comprueba que salen iguales; luego escribe la cadena completa.

---

## 4. Apartado b): implementar `from` y `to`

### Plantilla

In [ ]:
class Iso[A, B] extends Isomorphic[TipoIzquierdo, TipoDerecho]:

    def from(x: TipoIzquierdo): TipoDerecho =
        ???

    def to(y: TipoDerecho): TipoIzquierdo =
        ???

`class Iso[A, B] extends Isomorphic[...]` se lee: "defino una clase `Iso` (genérica en `A`
y `B`) que cumple la interfaz `Isomorphic` para estos dos tipos". Después del `:`, con
indentación, van los métodos.

### Recetas según la forma del tipo

1. **Si la entrada es un `Either` u `Option`** → `match` con un caso por cada variante
   (desglosa también los `Either` anidados: `Left(Left(a))`, `Left(Right(b))`, `Right(c)`…).
2. **Si la entrada es una tupla** → desestructúrala: `case (x, y) => ...` o usa `t._1`, `t._2`.
3. **Si tienes que devolver una función** → escribe una lambda (`x => ...`) o una
   función hecha de `case` (`{ case None => ...; case Some(a) => ... }`).
4. **Si la entrada es una función con dominio pequeño** (`Boolean`, `Unit`, `Option`,
   `Either`) → **evalúala en todos los valores posibles** de su dominio:
   `f(false)` y `f(true)`; `f(())`; `f(None)` y `a => f(Some(a))`; `a => f(Left(a))` y `b => f(Right(b))`.
5. **Comprueba** que cada caso de `from` vuelve a su caso correspondiente en `to`. El error
   más típico es cruzar dos casos (y entonces la ley falla).

Truco para no liarse: haz una tabla de correspondencias antes de escribir.

```
Izquierda            <->  Derecha
Left((b, Left(a)))   <->  (Some(b), Right(a))
Left((b, Right(c)))  <->  (Some(b), Left(c))
Right(Left(a))       <->  (None, Right(a))
Right(Right(c))      <->  (None, Left(c))
```

Luego `from` es la tabla leída de izquierda a derecha y `to` de derecha a izquierda.

---

## 5. Ejemplos resueltos

### Ejemplo 1 (25-26 Extraordinaria): `Option[A] => B` ≅ `(B, A => B)`

a) La cadena está en la sección 3.

b)

In [ ]:
class Iso[A, B] extends Isomorphic[Option[A] => B, (B, A => B)]:

    def from(f: Option[A] => B): (B, A => B) =
        (f(None), a => f(Some(a)))     // receta 4: evalúo f en None y en Some(a)

    def to(p: (B, A => B)): Option[A] => B =
        case None    => p._1           // receta 3: devuelvo una función hecha de case
        case Some(a) => p._2(a)

En `to`, el cuerpo `case None => ...; case Some(a) => ...` es directamente una función de
tipo `Option[A] => B`.

### Ejemplo 2 (25-26 Vicálvaro): `Either[(B, Either[A, C]), Either[A, C]]` ≅ `(Option[B], Either[C, A])`

a)

```
|Either[(B, Either[A, C]), Either[A, C]]| =
|(B, Either[A, C])| + |Either[A, C]| =
|B| * |Either[A, C]| + |Either[A, C]| =
(|B| + 1) * |Either[A, C]| =          // sacar factor común
(|B| + 1) * (|A| + |C|) =
(|B| + 1) * (|C| + |A|) =             // conmutativa
|Option[B]| * |Either[C, A]| =
|(Option[B], Either[C, A])|
```

b) Con la tabla de la sección 4:

In [ ]:
class Iso[A, B, C] extends Isomorphic[Either[(B, Either[A, C]), Either[A, C]], (Option[B], Either[C, A])]:

    def from(e: Either[(B, Either[A, C]), Either[A, C]]): (Option[B], Either[C, A]) =
        e match
            case Left((b, Left(a)))  => (Some(b), Right(a))
            case Left((b, Right(c))) => (Some(b), Left(c))
            case Right(Left(a))      => (None, Right(a))
            case Right(Right(c))     => (None, Left(c))

    def to(t: (Option[B], Either[C, A])): Either[(B, Either[A, C]), Either[A, C]] =
        t match
            case (None, Left(c))     => Right(Right(c))
            case (None, Right(a))    => Right(Left(a))
            case (Some(b), Left(c))  => Left((b, Right(c)))
            case (Some(b), Right(a)) => Left((b, Left(a)))

### Ejemplo 3 (25-26 Vicálvaro Bis, "demuestra o refuta"):

### `Either[Either[(A, B), (B, C)], Either[A, C]]` ≅ `(Either[B, Unit], Either[C, A])`

Cuando el enunciado dice "**demuestra o refuta**", primero haz las cuentas:

```
Izquierda: |A|·|B| + |B|·|C| + |A| + |C|
Derecha:   (|B| + 1) · (|C| + |A|) = |B|·|C| + |B|·|A| + |C| + |A|
```

Salen iguales → **sí son isomorfos** y hay que implementarlo. (Si salieran distintos, lo
refutas: basta con un ejemplo con tipos concretos, por ejemplo `A = B = C = Unit`, donde las
cardinalidades no coinciden.)

> Nota: la solución del archivo `Examen25-26-Ordinaria-Vicálvaro-Bis.ipynb` copia por error
> la del examen de Vicálvaro (otros tipos). Esta es la correcta para los tipos del enunciado.

`Either[B, Unit]` hace el papel de `Option[B]` (`Left(b)` ≈ `Some(b)`, `Right(())` ≈ `None`).

In [ ]:
class Iso[A, B, C] extends Isomorphic[Either[Either[(A, B), (B, C)], Either[A, C]], (Either[B, Unit], Either[C, A])]:

    def from(e: Either[Either[(A, B), (B, C)], Either[A, C]]): (Either[B, Unit], Either[C, A]) =
        e match
            case Left(Left((a, b)))  => (Left(b), Right(a))
            case Left(Right((b, c))) => (Left(b), Left(c))
            case Right(Left(a))      => (Right(()), Right(a))
            case Right(Right(c))     => (Right(()), Left(c))

    def to(t: (Either[B, Unit], Either[C, A])): Either[Either[(A, B), (B, C)], Either[A, C]] =
        t match
            case (Left(b), Right(a))  => Left(Left((a, b)))
            case (Left(b), Left(c))   => Left(Right((b, c)))
            case (Right(()), Right(a)) => Right(Left(a))
            case (Right(()), Left(c))  => Right(Right(c))

### Ejemplo 4 (25-26 Móstoles, de los difíciles):

### `Boolean => Either[A, B]` ≅ `Either[Boolean => A, Either[(Boolean, A, B), Boolean => B]]`

a)

```
|Boolean => Either[A, B]| =
|Either[A, B]| ^ |Boolean| =
(|A| + |B|) ^ 2 =
|A|^2 + 2*|A|*|B| + |B|^2 =
|Boolean => A| + |(Boolean, A, B)| + |Boolean => B| =
|Either[Boolean => A, Either[(Boolean, A, B), Boolean => B]]|
```

(`|A|^2` es `Boolean => A`; `2·|A|·|B|` es `(Boolean, A, B)`; `|B|^2` es `Boolean => B`.)

b) Una función `t: Boolean => Either[A, B]` queda determinada por dos valores: `t(false)` y
`t(true)`. Cada uno es `Left` o `Right` → 4 combinaciones:

- los dos `Left` → es una `Boolean => A`
- uno `Left` y otro `Right` → guardo **la a, la b y un Boolean que diga dónde estaba la a**
- los dos `Right` → es una `Boolean => B`

In [ ]:
class Iso[A, B] extends Isomorphic[
        Boolean => Either[A, B],
        Either[Boolean => A, Either[(Boolean, A, B), Boolean => B]]]:

    def from(t: Boolean => Either[A, B]): Either[Boolean => A, Either[(Boolean, A, B), Boolean => B]] =
        (t(false), t(true)) match
            case (Left(a1), Left(a2))   => Left({ case false => a1; case true => a2 })
            case (Left(a), Right(b))    => Right(Left((false, a, b)))   // la a estaba en false
            case (Right(b), Left(a))    => Right(Left((true, a, b)))    // la a estaba en true
            case (Right(b1), Right(b2)) => Right(Right({ case false => b1; case true => b2 }))

    def to(e: Either[Boolean => A, Either[(Boolean, A, B), Boolean => B]]): Boolean => Either[A, B] =
        e match
            case Left(f)                 => b => Left(f(b))
            case Right(Left((false, a, b))) => { case false => Left(a); case true => Right(b) }
            case Right(Left((true, a, b)))  => { case false => Right(b); case true => Left(a) }
            case Right(Right(g))         => b => Right(g(b))

### Ejemplo 5 (23-24 Vicálvaro): `(Unit => Boolean) => Unit => A` ≅ `(A, A)`

Cuentas: `|Unit => A| ^ |Unit => Boolean|` = `(|A|^1) ^ (2^1)` = `|A|^2` = `|(A, A)|`.

In [ ]:
class Iso[A] extends Isomorphic[(Unit => Boolean) => Unit => A, (A, A)]:

    def from(t: (Unit => Boolean) => Unit => A): (A, A) =
        (t(_ => false)(()), t(_ => true)(()))     // pruebo t con la función "siempre false" y "siempre true"

    def to(t: (A, A)): (Unit => Boolean) => Unit => A =
        f => _ => if f(()) then t._2 else t._1

`_ => false` es una función `Unit => Boolean` que ignora su argumento. `(())` es pasarle el
único valor de `Unit`.

### Ejemplo 6: funciones polimórficas `[C] => ...` (24-25 Móstoles y Vicálvaro)

Scala 3 tiene un tipo especial para "una función que funciona **para cualquier tipo C**":

```scala
[C] => (A => B => C) => C
```

Se lee: "para todo tipo C, si me das una función `A => B => C`, te devuelvo una `C`".

- **Usarla** (si tienes `t` de ese tipo): primero dices qué `C` quieres entre corchetes y
  luego le pasas el argumento: `t[(A, B)](a => b => (a, b))`.
- **Construirla**: `[C] => (f: A => B => C) => ...` (hay que escribir el tipo del parámetro).

`(A, B)` ≅ `[C] => (A => B => C) => C`:

In [ ]:
class Iso[A, B] extends Isomorphic[[C] => (A => B => C) => C, (A, B)]:

    def from(t: [C] => (A => B => C) => C): (A, B) =
        t[(A, B)](a => b => (a, b))         // elijo C = (A, B) y le paso "construir el par"

    def to(t: (A, B)): [C] => (A => B => C) => C =
        [C] => (f: A => B => C) => f(t._1)(t._2)

`Either[A, B]` ≅ `[C] => (A => C) => (B => C) => C`:

In [ ]:
class Iso[A, B] extends Isomorphic[[C] => (A => C) => (B => C) => C, Either[A, B]]:

    def from(t: [C] => (A => C) => (B => C) => C): Either[A, B] =
        t[Either[A, B]](a => Left(a))(b => Right(b))

    def to(t: Either[A, B]): [C] => (A => C) => (B => C) => C =
        t match
            case Left(a)  => [C] => (f: A => C) => (g: B => C) => f(a)
            case Right(b) => [C] => (f: A => C) => (g: B => C) => g(b)

Truco general: **para sacar el dato de una función polimórfica, elige como `C` el propio
tipo que quieres obtener y pásale los constructores** (`(a, b)`, `Left`, `Right`,
`identity`…).

### Ejemplo 7: lo mismo con un `trait` (22-23 Vicálvaro)

En exámenes de Scala 2 se hacía con un trait con un método genérico:

In [ ]:
trait Cont[A]:
    def apply[X]: (A => X) => X

class Iso[A] extends Isomorphic[Cont[A], A]:
    def from(f: Cont[A]): A =
        f[A](identity)                 // X = A, y le paso la identidad
    def to(a: A): Cont[A] =
        new Cont[A] {                  // objeto anónimo que implementa el trait
            def apply[X]: (A => X) => X = f => f(a)
        }

### Ejemplo 8: listas (22-23 Móstoles): `List[A]` ≅ `1 + A + A² · List[A]`

In [ ]:
class Iso[A] extends Isomorphic[Either[Unit, Either[A, (A, A, List[A])]], List[A]]:
    def from(e: Either[Unit, Either[A, (A, A, List[A])]]): List[A] =
        e match
            case Left(_)                   => Nil
            case Right(Left(a))            => a :: Nil
            case Right(Right((h1, h2, t))) => h1 :: h2 :: t

    def to(l: List[A]): Either[Unit, Either[A, (A, A, List[A])]] =
        l match
            case Nil           => Left(())
            case a :: Nil      => Right(Left(a))
            case h1 :: h2 :: t => Right(Right((h1, h2, t)))

---

## 6. Errores típicos

- **Poner la potencia al revés.** `A => B` es `|B|^|A|`, no `|A|^|B|`.
- **Cruzar casos** entre `from` y `to` (por ejemplo devolver `Left` donde tocaba `Right`).
  La función compila pero **no** es inversa. Haz siempre la tabla de correspondencias.
- **Olvidar un caso** de un `Either` anidado.
- **Escribir `Left(false, a, b)` en vez de `Left((false, a, b))`.** Scala a veces lo acepta,
  pero pon los paréntesis dobles para crear la tupla explícitamente.
- **`Right()` en vez de `Right(())`.** El valor de `Unit` es `()`, así que va dentro: `Right(())`.

---

## 7. Ejercicios para practicar

### De exámenes reales

| Isomorfismo | Examen |
|---|---|
| `(A Either B Either C) => D` ≅ `(A => D, B => D, C => D)` | 24-25 Extraordinaria |
| `(Either[A, B], Either[C, D])` ≅ `Either4[(A,C), (A,D), (B,C), (B,D)]` | 23-24 Móstoles |
| `Either[Unit => (X, Y), (Z, (X, Either[Nothing, Unit]))]` ≅ `Either[(X, Y), (X, Z)]` | 23-24 Extraordinaria |
| `(Either[P, Q], Either[R, S])` ≅ `Either[(P,R), Either[(P,S), Either[(Q,R), (Q,S)]]]` | 22-23 Extraordinaria |
| `((Y, Either[Z, V])) => X` ≅ `(Z => Y => X, V => Y => X)` | 22-23 Extraordinaria |
| `(A => R, Unit => R)` ≅ `Option[A] => R` | 22-23 Vicálvaro Especial |
| `Option[Nel[A]]` ≅ `List[A]`, con `Nel[A] = (A, List[A])` | 21-22 Extraordinaria |
| `(1+Y+Z+Y·Z)^X` ≅ `(Y+1)^X · (Z+1)^X` | 21-22 Móstoles |
| `(1+1)^X` ≅ `Boolean^X` | 21-22 Vicálvaro |
| `(X+1)²` ≅ `X² + 2X + 1` | 20-21 Ordinaria |
| `(Y·Z)^X` ≅ `Y^X · Z^X` | 20-21 Extraordinaria |

Además: `PF-2/topic3/3.4 Isomorphisms-Exercises.ipynb` tiene más ejercicios del curso.

### Ejercicios nuevos

1. `(A, Either[B, C])` ≅ `Either[(A, B), (A, C)]`
2. `Either[B, C] => A` ≅ `(B => A, C => A)`
3. `Boolean => A` ≅ `(A, A)`

<details>
<summary>Soluciones</summary>

```scala
// 1
class Iso1[A, B, C] extends Isomorphic[(A, Either[B, C]), Either[(A, B), (A, C)]]:
    def from(t: (A, Either[B, C])): Either[(A, B), (A, C)] =
        t match
            case (a, Left(b))  => Left((a, b))
            case (a, Right(c)) => Right((a, c))
    def to(e: Either[(A, B), (A, C)]): (A, Either[B, C]) =
        e match
            case Left((a, b))  => (a, Left(b))
            case Right((a, c)) => (a, Right(c))

// 2
class Iso2[A, B, C] extends Isomorphic[Either[B, C] => A, (B => A, C => A)]:
    def from(f: Either[B, C] => A): (B => A, C => A) =
        (b => f(Left(b)), c => f(Right(c)))
    def to(t: (B => A, C => A)): Either[B, C] => A =
        case Left(b)  => t._1(b)
        case Right(c) => t._2(c)

// 3
class Iso3[A] extends Isomorphic[Boolean => A, (A, A)]:
    def from(f: Boolean => A): (A, A) =
        (f(false), f(true))
    def to(t: (A, A)): Boolean => A =
        b => if b then t._2 else t._1
```

</details>

Siguiente: **04-recursion-y-foldRight.ipynb**.